# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [1]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [3]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [4]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'11f7f5cf-1976-49fa-b97d-37ae2b82e354': {'version…

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [5]:
from textwrap import dedent

PROMPT_V1 = dedent("""
You are PRIZEBOT at a neon arcade prize counter.

SOURCE OF TRUTH (only these exist):
- Sticker Pack — 20 tickets
- Candy Bar — 35 tickets
- LED Keychain — 80 tickets
- Mini Plush — 120 tickets
- Foam Katana — 160 tickets
- Mystery Box — 200 tickets

RULES:
- Welcome first.
- Never invent prizes or prices. If asked for something else, say unavailable + suggest 1–2 listed prizes.
- If the user doesn’t say how many tickets they have, ask.
- Confirm quantities.
- Before “finalizing”: summarize items + total tickets + tickets remaining, then ask to confirm.

Style: short, friendly, cashier energy.
""").strip()

In [6]:
PROMPT_V2 = dedent("""
You are CLERK-7 on a space station parts desk.

CATALOG (only these exist):
- Oxygen Filter (S/M/L) — 15 / 25 / 40 credits
- Coolant Cell (Standard/XL) — 30 / 55 credits
- Circuit Patch Kit — 12 credits
- Hull Sealant Tube — 18 credits
- Power Coupler — 22 credits

RULES:
- Welcome first.
- Never invent parts or prices.
- Always confirm required specs (size/type) when applicable.
- Ask ONE bundled question when info is missing.
- Before checkout: itemized list + total credits, then ask “Confirm?”

Tone: clear, efficient, light sci-fi flavor (not heavy roleplay).
""").strip()

In [7]:
PROMPT_V3 = dedent("""
You are FESTIVALBOT helping build a film-festival schedule.

PROGRAM (only these exist):
- 12:00 — Glass River (90 min)
- 13:30 — Signal Drift (105 min)
- 15:45 — Paper Moons (80 min)
- 17:15 — Red Static (110 min)
- 19:30 — After the Flood (95 min)

RULES:
- Welcome first.
- Never invent films or times.
- Ask up to 2 questions (start time / must-sees / max films).
- Build a schedule with no overlaps (use durations).
- Before finalizing: summarize schedule + total viewing time, then ask to confirm.

Tone: friendly planner, concise.
""").strip()

In [ ]:
pn.extension()

def load_version(system_prompt, title="Vertical Chat"):
    global context, panels, client_prompt, button_conversation, interactive_conversation, dashboard

    context = [{"role": "system", "content": system_prompt}]
    panels = []

    client_prompt = pn.widgets.TextInput(value="Hi", placeholder="Enter text here…")
    button_conversation = pn.widgets.Button(name="talk")

    # uses your existing function from the notebook:
    interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

    dashboard = pn.Column(
        pn.pane.Markdown(f"### {title}"),
        client_prompt,
        pn.Row(button_conversation),
        pn.panel(interactive_conversation, loading_indicator=True),
    )
    return dashboard
print("Ready. Call: load_version(PROMPT_V1, 'Version 1 — Arcade') etc.")

Ready. Call: load_version(PROMPT_V1, 'Version 1 — Arcade') etc.


In [9]:
load_version(PROMPT_V1, "Version 1 — Arcade Prizebot")

BokehModel(combine_events=True, render_bundle={'docs_json': {'5f48f772-d5d2-4e00-bbd7-070c7934a48b': {'version…

### Findings / Results (What happened)

- All three prompt versions stayed grounded in their “source of truth” lists (prizes / parts catalog / film program).
- When I tested **prompt-mixing / bait requests** (asking for an item that sounds plausible but doesn’t exist), the bots **did not invent** the item.
- **Arcade Prizebot example:** I asked for a **“Mystery Katana”** (a mash-up of *Foam Katana* + *Mystery Box*). The bot correctly said that item is **not available**, then redirected me to valid options (**Foam Katana** or **Mystery Box**).
- Overall, the bots followed the intended flow: clarify missing details → stay within the allowed options → summarize and confirm rather than guessing.

### Why it worked

- Each prompt created a **closed world** by explicitly stating “ONLY these exist.” That prevented the model from blending items or generating “reasonable-sounding” new ones.
- The prompts also included a clear fallback behavior: **refuse unavailable requests + offer alternatives**. This kept the conversation moving without making anything up.
- The “ask clarifying questions when needed” rule reduced ambiguity, which also reduces hallucinations (the bot doesn’t need to guess missing details).

### What I learned

- Reliability improves a lot when a prompt includes:
  1) a clear **source of truth** (a fixed list/table/program),
  2) explicit **anti-hallucination rules** (“don’t invent items or prices”),
  3) a **repair policy** for invalid requests (politely refuse + suggest valid alternatives).
- Testing with “blended” requests (like “Mystery Katana”) is a good way to check if the prompt is actually constraining the assistant.

### Possible improvements

- Add an explicit rule: “When refusing an unavailable request, always suggest the closest matching valid options.”
- Add a short confirmation question after redirecting: “Do you want option A or option B?”